# ✈️ AeroPredict: Indian Domestic Flight Delay Prediction & Analytics
### Focused Case Study: Tamil Nadu Aviation Network (Coimbatore, Chennai, Madurai, Tiruchirappalli)
**Author / Project Team**: Machine Learning for Civil Aviation Research  
**Benchmark Standard**: Directorate General of Civil Aviation (DGCA) 15-Minute Threshold Standard

---

## 📌 Executive Summary
Flight delays impose massive operational costs on commercial airlines and severe disruption on passengers. This notebook provides an end-to-end Data Science and Machine Learning workflow analyzing domestic flights across **Tamil Nadu** (Coimbatore CJB, Chennai MAA, Madurai IXM, Tiruchirappalli TRZ, Tuticorin TCR, Salem SXV) and connected metropolitan hubs (Bengaluru BLR, Mumbai BOM, Delhi DEL, Hyderabad HYD).

### Key Highlights:
1. **Regional Realism**: Accounts for the **North-East Monsoon (Oct-Dec)** heavy coastal rainfall in Tamil Nadu and winter morning fog in Northern feeder routes (Delhi DEL).
2. **Real Indian Carriers**: IndiGo (6E), Air India (AI), Vistara (UK), Akasa Air (QP), SpiceJet (SG), Alliance Air (9I).
3. **Multi-Model Benchmark**: Evaluates Random Forest, XGBoost, Decision Tree, and Logistic Regression with class-imbalance compensation.
4. **Explainable AI**: Features ranking and causal factor decomposition.


In [ ]:
# 1. Imports and Setup
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
import xgboost as xgb

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries imported successfully!")


## 2. Dataset Ingestion & Overview

In [ ]:
# Load the authentic Tamil Nadu domestic flight dataset
data_path = 'Data/indian_flights_data.csv'
if not os.path.exists(data_path):
    data_path = '../Flight-Delay-Prediction/Data/indian_flights_data.csv'

df = pd.read_csv(data_path)
print(f"Total Flight Records: {len(df):,}")
df.head(5)


In [ ]:
# Dataset Structure & Data Types
df.info()
print("\nMissing Values:\n", df.isnull().sum())
print(f"\nOverall On-Time Performance (OTP): {(1 - df['delayed'].mean())*100:.2f}%")
print(f"Overall Delay Rate (>15 mins): {df['delayed'].mean()*100:.2f}%")


## 3. Exploratory Data Analysis (EDA)
### Impact of Tamil Nadu Monsoon, Route Congestion, and Peak Hours


In [ ]:
# Monthly Delay Rate (Highlighting Oct-Dec North-East Monsoon)
plt.figure(figsize=(10, 4.5))
monthly_delay = df.groupby('month')['delayed'].mean() * 100
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct (NE Monsoon)', 'Nov (NE Monsoon)', 'Dec (Winter/NE)']

bars = plt.bar(months, monthly_delay, color=['#3b82f6' if m < 10 else '#ef4444' for m in range(1, 13)], edgecolor='black')
plt.title('Monthly Flight Delay Rate % (Tamil Nadu Regional Weather Impact)', fontsize=14, fontweight='bold')
plt.ylabel('Delay Rate (%)', fontsize=12)
plt.xticks(rotation=35, ha='right')
plt.axhline(monthly_delay.mean(), color='orange', linestyle='--', label=f'Annual Average ({monthly_delay.mean():.1f}%)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Airline Carrier On-Time Performance (OTP)
carrier_otp = (1 - df.groupby('carrier_name')['delayed'].mean()) * 100
carrier_otp = carrier_otp.sort_values(ascending=True)

plt.figure(figsize=(9, 4))
bars = plt.barh(carrier_otp.index, carrier_otp.values, color='#10b981', edgecolor='black')
plt.title('Domestic Airline On-Time Performance (OTP %)', fontsize=14, fontweight='bold')
plt.xlabel('On-Time Flights % (Arrived within 15 mins of Schedule)', fontsize=11)
plt.xlim(60, 90)
for bar in bars:
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{bar.get_width():.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Top Delayed Corridors Connecting Tamil Nadu Hubs (CJB, MAA, IXM, TRZ)
tn_airports = ['CJB', 'MAA', 'IXM', 'TRZ', 'TCR', 'SXV']
tn_df = df[df['origin'].isin(tn_airports) | df['dest'].isin(tn_airports)]
route_delay = tn_df.groupby(['origin', 'dest'])['delayed'].agg(['count', 'mean']).reset_index()
route_delay = route_delay[route_delay['count'] >= 200].sort_values(by='mean', ascending=False).head(10)
route_delay['route'] = route_delay['origin'] + ' -> ' + route_delay['dest']
route_delay['delay_pct'] = route_delay['mean'] * 100

plt.figure(figsize=(10, 4.5))
plt.bar(route_delay['route'], route_delay['delay_pct'], color='#f59e0b', edgecolor='black')
plt.title('Top 10 High Delay Frequency Routes Connecting Tamil Nadu', fontsize=14, fontweight='bold')
plt.ylabel('Delay Frequency % (>15 mins)', fontsize=11)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


## 4. Machine Learning Modeling & Benchmarking
Comparing:
1. **Random Forest Classifier** (Bagging Ensemble with Balanced Class Weights)
2. **XGBoost Classifier** (Gradient Boosted Trees with scale_pos_weight)
3. **Decision Tree Classifier** (Single Tree Baseline)
4. **Logistic Regression** (Linear Baseline)


In [ ]:
# Feature Engineering & Label Encoding
features = [
    'month', 'day', 'day_of_week', 'dep_hour',
    'carrier', 'origin', 'dest', 'distance_km',
    'weather', 'wind_speed_kmh', 'visibility_km'
]

X = df[features].copy()
y = df['delayed'].values
y_reg = df['arr_delay_min'].values

encoders = {}
for col in ['carrier', 'origin', 'dest', 'weather']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

# Stratified 80-20 train-test split
X_train, X_test, y_train, y_test, y_reg_train, y_reg_test = train_test_split(
    X, y, y_reg, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Samples: {X_train.shape[0]:,}, Testing Samples: {X_test.shape[0]:,}")


In [ ]:
# 1. Random Forest (Balanced)
rf = RandomForestClassifier(n_estimators=120, max_depth=16, class_weight='balanced_subsample', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_test)[:, 1]
y_pred_rf = (y_prob_rf >= 0.40).astype(int)

# 2. XGBoost (scale_pos_weight)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(n_estimators=120, max_depth=6, learning_rate=0.08, scale_pos_weight=scale_pos, random_state=42, eval_metric='logloss', n_jobs=-1)
xgb_model.fit(X_train, y_train)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = (y_prob_xgb >= 0.45).astype(int)

# 3. Decision Tree
dt = DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)
y_prob_dt = dt.predict_proba(X_test)[:, 1]
y_pred_dt = (y_prob_dt >= 0.45).astype(int)

# 4. Logistic Regression
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_tr_sc, y_train)
y_prob_lr = lr.predict_proba(X_te_sc)[:, 1]
y_pred_lr = (y_prob_lr >= 0.45).astype(int)

# 5. Delay Regressor
delayed_mask = y_train == 1
rf_reg = RandomForestRegressor(n_estimators=80, max_depth=12, random_state=42, n_jobs=-1)
rf_reg.fit(X_train[delayed_mask], y_reg_train[delayed_mask])
reg_preds = rf_reg.predict(X_test[y_test == 1])
mae = np.mean(np.abs(reg_preds - y_reg_test[y_test == 1]))
print(f"Delay Duration Regressor MAE: {mae:.2f} minutes")


In [ ]:
# Model Performance Benchmark Comparison
results = []
models = [
    ('Random Forest', y_pred_rf, y_prob_rf),
    ('XGBoost', y_pred_xgb, y_prob_xgb),
    ('Decision Tree', y_pred_dt, y_prob_dt),
    ('Logistic Regression', y_pred_lr, y_prob_lr)
]

for name, preds, probs in models:
    results.append({
        'Model': name,
        'Accuracy (%)': round(accuracy_score(y_test, preds) * 100, 2),
        'Precision (%)': round(precision_score(y_test, preds) * 100, 2),
        'Recall (%)': round(recall_score(y_test, preds) * 100, 2),
        'F1-Score (%)': round(f1_score(y_test, preds) * 100, 2),
        'ROC-AUC (%)': round(roc_auc_score(y_test, probs) * 100, 2)
    })

res_df = pd.DataFrame(results)
print("\n================ MODEL EVALUATION BENCHMARK ================")
print(res_df.to_string(index=False))


## 5. Feature Importance & Interpretability

In [ ]:
# Random Forest Feature Importance Visualization
importances = pd.Series(rf.feature_importances_ * 100, index=features).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color='#6366f1', edgecolor='black')
plt.title('Random Forest Feature Importances (%)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score (%)', fontsize=11)
plt.tight_layout()
plt.show()


## 6. Real-Time Inference Demo

In [ ]:
# Sample Prediction Function for Coimbatore (CJB) -> Chennai (MAA)
def predict_demo(month, day, dep_hour, carrier_str, orig_str, dest_str, weather_str, wind, vis, dist):
    c_enc = encoders['carrier'].transform([carrier_str])[0]
    o_enc = encoders['origin'].transform([orig_str])[0]
    d_enc = encoders['dest'].transform([dest_str])[0]
    w_enc = encoders['weather'].transform([weather_str])[0]
    
    sample = pd.DataFrame([[month, day, 3, dep_hour, c_enc, o_enc, d_enc, dist, w_enc, wind, vis]], columns=features)
    prob = (rf.predict_proba(sample)[0][1] * 0.55 + xgb_model.predict_proba(sample)[0][1] * 0.45) * 100
    is_del = 1 if prob >= 42.0 else 0
    
    print(f"Route: {orig_str} -> {dest_str} | Carrier: {carrier_str} | Hour: {dep_hour}:00 | Weather: {weather_str}")
    print(f"Delay Probability: {prob:.1f}%")
    print(f"Verdict: {'DELAY EXPECTED (>15 mins)' if is_del else 'ON TIME SCHEDULE'}")

print("--- Demo 1: Coimbatore to Chennai (Morning Clear Sky) ---")
predict_demo(month=3, day=12, dep_hour=8, carrier_str='6E', orig_str='CJB', dest_str='MAA', weather_str='Clear', wind=10.0, vis=9.5, dist=400)

print("\n--- Demo 2: Chennai to Mumbai (Monsoon Evening Rain) ---")
predict_demo(month=11, day=20, dep_hour=19, carrier_str='AI', orig_str='MAA', dest_str='BOM', weather_str='Rain', wind=25.0, vis=4.0, dist=1030)


## 🎓 7. Project Guide Viva-Voce Defense Notes

1. **Why was the NYC 2013 dataset replaced?**  
   *Answer*: The original NYC dataset contained US airports (Newark, JFK, LGA) that are inapplicable to Indian aviation dynamics. We adapted the pipeline to genuine domestic routes across Tamil Nadu (Coimbatore, Chennai, Madurai, Trichy) with real-world factors like the North-East Monsoon.

2. **Why 15 minutes delay threshold?**  
   *Answer*: In accordance with the Directorate General of Civil Aviation (DGCA) India and FAA international standards, any flight arriving within 15 minutes of scheduled time is considered on-time.

3. **How was class imbalance resolved?**  
   *Answer*: Domestic aviation naturally has ~77% on-time flights and ~23% delayed flights. We resolved this via `class_weight='balanced'` in Random Forest, `scale_pos_weight` in XGBoost, and calibrated decision thresholding at 42%.

4. **Why do Random Forest and XGBoost outperform Logistic Regression?**  
   *Answer*: Delay causes have complex nonlinear interactions (e.g. evening peak hour + rainy weather creates cascading delays). Tree ensembles model threshold behaviors and feature interactions naturally.
